In [ ]:
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import gamma, norm


def calculate_spi(precip_series, scale=1):
    """Calculate Standardized Precipitation Index (SPI) for a given precipitation series."""
    precip_rolling = precip_series.rolling(window=scale).sum()
    precip_rolling = precip_rolling.dropna()
    shape, loc, scale_param = gamma.fit(precip_rolling, floc=0)
    gamma_cdf = gamma.cdf(precip_rolling, shape, loc=loc, scale=scale_param)
    spi_values = norm.ppf(gamma_cdf)
    return pd.Series(spi_values, index=precip_rolling.index)


def calculate_spi_for_regions(zambia_rain_df):
    """Calculate 1-month and 3-month SPI for all regions in the dataset."""
    spi_results_df = pd.DataFrame()
    admin2_names = zambia_rain_df['admin2_name'].unique()
    for admin2 in admin2_names:
        admin2_cleaned = admin2.replace(" ", "_").replace("-", "_")
        region_data = zambia_rain_df[zambia_rain_df['admin2_name'] == admin2][['date', 'precipitation']]
        region_data.set_index('date', inplace=True)
        region_data = region_data.groupby(region_data.index).mean()
        spi_1_month = calculate_spi(region_data['precipitation'], scale=1)
        spi_1_month.name = f'{admin2_cleaned}_1_month'
        spi_3_month = calculate_spi(region_data['precipitation'], scale=3)
        spi_3_month.name = f'{admin2_cleaned}_3_month'
        region_spi_df = pd.concat([spi_1_month, spi_3_month], axis=1)
        spi_results_df = pd.concat([spi_results_df, region_spi_df], axis=1)
    return spi_results_df


def plot_spi_dropdown(spi_results_df):
    """
    Generate a Plotly figure with a dropdown menu to switch between SPI plots for different regions and timescales.
    """
    fig = go.Figure()

    # Add traces for each SPI time scale
    for region_time_scale in spi_results_df.columns:
        spi_series = spi_results_df[region_time_scale].dropna()
        fig.add_trace(
            go.Scatter(
                x=spi_series.index,
                y=spi_series.values,
                mode="lines",
                name=region_time_scale,
                visible=False
            )
        )

    # Show the first trace by default
    if len(fig.data) > 0:
        fig.data[0].visible = True

    # Add reference lines for drought categories
    fig.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Neutral (SPI=0)")
    fig.add_hline(y=-1, line_dash="dash", line_color="orange", annotation_text="Mild Drought (SPI=-1)")
    fig.add_hline(y=-1.5, line_dash="dash", line_color="red", annotation_text="Moderate Drought (SPI=-1.5)")
    fig.add_hline(y=-2, line_dash="dash", line_color="darkred", annotation_text="Severe Drought (SPI=-2)")

    # Dropdown menu to switch between regions and timescales
    fig.update_layout(
        updatemenus=[
            dict(
                buttons=[
                    dict(
                        method="update",
                        label=region_time_scale,
                        args=[
                            {"visible": [region_time_scale == trace.name for trace in fig.data]},
                            {"title": f"SPI for {region_time_scale}"}
                        ]
                    ) for region_time_scale in spi_results_df.columns
                ],
                direction="down",
                x=0.5,
                xanchor="center",
                y=1.2,
                yanchor="top",
                showactive=True
            )
        ],
        title="Standardized Precipitation Index (SPI)",
        xaxis_title="Date",
        yaxis_title="SPI Value",
        template="plotly_white",
        height=600
    )
    return fig
